In [ ]:
#Use this script to generate train/dev/test splits

In [1]:
import warnings
warnings.filterwarnings("ignore", message="unknown class")

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

# Load the formatted data
df = pd.read_csv(
    "pilot_split.csv",
    dtype=str,                 # keep grams as strings
    keep_default_na=False,     # don't auto-treat "null"/"NA" as NaN
    na_values=[]               # no extra NA markers
)

# Features (X) and labels (y)
X = df[["TokenID", "Exp", "Res", "Univ", "DTAM", "Evid", "TAML", "NegPol", "akt_activity", "akt_accomp", "akt_achiev", "akt_state", "akt_semel"]]  # adjust as needed; taken out for now: , "ant_complete", "ant_initial", "ant_none"
y = df[["eng_gram", "deu_gram", "cmn_gram"]] #taken out for now: , "kat_gram", "jpn_gram"

# Filter label classes that occur fewer than twice
y = df[["eng_gram", "deu_gram", "cmn_gram"]] #taken out for now: , "kat_gram", "jpn_gram"
X = df[["TokenID", "Exp", "Res", "Univ", "DTAM", "Evid", "TAML", "NegPol",
        "akt_activity", "akt_accomp", "akt_achiev", "akt_state", "akt_semel"]] #taken out for now: , "ant_complete", "ant_initial", "ant_none"

label_counts = y.value_counts()
valid_labels = label_counts[label_counts >= 2].index

# Define which gram columns to transform
gram_cols = ["eng_gram", "deu_gram", "cmn_gram"]

# Convert space-separated strings into lists — apply directly to y
for col in gram_cols:
    y[col] = y[col].apply(lambda x: [] if x == "n_a" else x.split())

# Quick check
print(y["deu_gram"].head(10).tolist())
print(type(y["deu_gram"].iloc[0]))

# Proceed with splitting -- 50/50
X_train, X_dev, y_train, y_dev = train_test_split(X, y, test_size=0.5, random_state=42)

# Proceed with splitting -- 50/25/25
#X_temp, X_test, y_temp, y_test = train_test_split(X_filtered, y_filtered, test_size=0.2, random_state=42)
#X_train, X_dev, y_train, y_dev = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Remove n_a rows separately per language
X_train_no_na = {}
X_dev_no_na = {}
y_train_no_na = {}
y_dev_no_na = {}

for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    mask_train = ~y_train[lang].apply(lambda lbls: lbls == [] or lbls == ['n_a'])
    mask_dev   = ~y_dev[lang].apply(lambda lbls: lbls == [] or lbls == ['n_a'])
    
    # Keep separate copies of filtered data for this language
    X_train_no_na[lang] = X_train.loc[mask_train].reset_index(drop=True)
    X_dev_no_na[lang]   = X_dev.loc[mask_dev].reset_index(drop=True)
    y_train_no_na[lang] = y_train.loc[mask_train, lang].reset_index(drop=True)
    y_dev_no_na[lang]   = y_dev.loc[mask_dev, lang].reset_index(drop=True)

# Remove rows where any language gram is n_a before training
#mask_no_na = ~y_train.applymap(lambda lbls: lbls == [] or lbls == ['n_a']).any(axis=1)
#X_train = X_train[mask_no_na].reset_index(drop=True)
#y_train = y_train[mask_no_na].reset_index(drop=True)

#mask_no_na_dev = ~y_dev.applymap(lambda lbls: lbls == [] or lbls == ['n_a']).any(axis=1)
#X_dev = X_dev[mask_no_na_dev].reset_index(drop=True)
#y_dev = y_dev[mask_no_na_dev].reset_index(drop=True)

X_train_model = X_train.drop(columns=["TokenID"])
X_dev_model   = X_dev.drop(columns=["TokenID"])

mlbs = {}
y_train_bin = {}
y_dev_bin   = {}

for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    mlb = MultiLabelBinarizer()
    y_train_bin[lang] = mlb.fit_transform(y_train_no_na[lang])  # <- use *_no_na
    y_dev_bin[lang]   = mlb.transform(y_dev_no_na[lang])        # <- use *_no_na
    mlbs[lang] = mlb
    print(f"{lang}: {len(mlb.classes_)} labels → {mlb.classes_[:10]}...")

[['prat', 'perfekt'], ['prat', 'perfekt'], ['perfekt'], ['perfekt'], ['prat', 'perfekt'], ['perfekt'], ['perfekt'], ['perfekt'], [], []]
<class 'list'>
eng_gram: 6 labels → ['might' 'past' 'perfect' 'prog' 'should' 'would']...
deu_gram: 5 labels → ['k1' 'k2' 'perfekt' 'prasens' 'prat']...
cmn_gram: 10 labels → ['guo' 'le' 'meiyou' 'nullmarked' 'qilai' 'suo_de' 'yi' 'yijing' 'zaojiu'
 'zaoyi']...


/var/folders/22/4lgkjkkn2vvb2m8ffgsjl7gh0000gn/T/ipykernel_47799/2123216697.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y[col] = y[col].apply(lambda x: [] if x == "n_a" else x.split())


In [2]:
print(len(df))

301


In [3]:
print(y["deu_gram"].head(10).tolist())
print(type(y["deu_gram"].iloc[0]))

[['prat', 'perfekt'], ['prat', 'perfekt'], ['perfekt'], ['perfekt'], ['prat', 'perfekt'], ['perfekt'], ['perfekt'], ['perfekt'], [], []]
<class 'list'>


In [6]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.multioutput import MultiOutputClassifier

# Train and predict for each language
for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    print(f"\n=== MLP Classification Report for {lang.upper()} ===")

    # === CHANGE 1: wrap MLP in MultiOutputClassifier ===
    #clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
    clf = MultiOutputClassifier(MLPClassifier(max_iter=500, random_state=42))
    # (You can reintroduce hidden_layer_sizes later if you want to tune depth.)

    # Train on filtered (no_n_a) data
    X_train_lang = X_train_no_na[lang].drop(columns=["TokenID"])
    X_dev_lang   = X_dev_no_na[lang].drop(columns=["TokenID"])
    y_train_lang = y_train_bin[lang]
    y_dev_lang   = y_dev_no_na[lang]

    # === CHANGE 2: fit and predict remain identical ===
    clf.fit(X_train_lang, y_train_lang)
    preds = clf.predict(X_dev_lang)

    # === CHANGE 3: keep using MultiLabelBinarizer inverse_transform ===
    pred_sets = mlbs[lang].inverse_transform(preds)
    true_sets = y_dev_lang

    # Keep aligned non-empty pairs
    pairs = [(t, p) for t, p in zip(true_sets, pred_sets) if t and p]
    true_vals = pd.Series([", ".join(t) for t, _ in pairs])
    pred_vals = pd.Series([", ".join(p) for _, p in pairs])

    # Summaries
    print("Shapes → true_vals:", true_vals.shape, "pred_vals:", pred_vals.shape)
    print("First 10 true:", true_vals.head(10).tolist())
    print("First 10 pred:", pred_vals.head(10).tolist())
    print("Manual accuracy:", (true_vals == pred_vals).mean())

    # Classification report
    print(classification_report(true_vals, pred_vals, zero_division=0))


=== MLP Classification Report for ENG_GRAM ===
Shapes → true_vals: (151,) pred_vals: (151,)
First 10 true: ['perfect', 'perfect', 'perfect', 'past, perfect', 'perfect', 'perfect', 'perfect', 'perfect', 'perfect', 'perfect']
First 10 pred: ['perfect', 'perfect', 'perfect', 'past, perfect', 'perfect', 'perfect', 'perfect', 'perfect', 'perfect', 'perfect']
Manual accuracy: 0.8874172185430463
                 precision    recall  f1-score   support

 could, perfect       0.00      0.00      0.00         3
  past, perfect       0.78      0.78      0.78        32
        perfect       0.94      0.99      0.96       110
 perfect, would       0.00      0.00      0.00         0
should, perfect       0.00      0.00      0.00         2
 would, perfect       0.00      0.00      0.00         4

       accuracy                           0.89       151
      macro avg       0.29      0.30      0.29       151
   weighted avg       0.85      0.89      0.87       151


=== MLP Classification Report for

In [ ]:
# sanity checks
assert {"eng_gram","deu_gram","cmn_gram"}.issubset(df.columns)
assert X_train.shape[0] == y_train.shape[0] and X_dev.shape[0] == y_dev.shape[0]
print("Train/dev sizes:", X_train.shape[0], X_dev.shape[0])

# ENG-specific masks: confirm we still have training rows
print("ENG train kept:", m_tr_eng.sum(), " / ", m_tr_eng.shape[0])
print("ENG dev kept:", m_de_eng.sum(), " / ", m_de_eng.shape[0])

# quick peek: tokenization looks sane
print("Sample y_train ENG tokens:", y_train.loc[m_tr_eng, "eng_gram"].head(3).tolist())